# 第 5 章 制御フロー: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/economics-programming/05-control-flow.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
demand_intercept = 100  # 需要 Q = a - bP の切片 a
demand_slope = 2        # 需要の傾き b
supply_intercept = 0    # 供給 Q = c + dP の切片 c
supply_slope = 3        # 供給の傾き d
initial_price = 0       # ワルラス調整の、最初の価格
walras_step = 0.1       # ワルラス調整の速さ（大きすぎると収束しない）
initial_quantity = 0    # マーシャル調整の、最初の数量
marshall_step = 0.1     # マーシャル調整の速さ

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 価格調整（ワルラス）と、数量調整（マーシャル）

In [ ]:
from lib.chapter01.demand import quantity_demanded
from lib.chapter03.schedule import quantity_supplied
from lib.chapter05.adjustment import marshall_path, walras_path
from lib.chapter07.surplus import equilibrium, inverse_demand

eq_price, eq_quantity = equilibrium(
    demand_intercept, demand_slope, supply_intercept, supply_slope
)
print(f"均衡: 価格 {eq_price:.2f}、数量 {eq_quantity:.2f}")

price_path = walras_path(
    lambda p: quantity_demanded(p, demand_intercept, demand_slope),
    lambda p: quantity_supplied(p, supply_intercept, supply_slope),
    initial_price,
    walras_step,
)
print(f"ワルラス調整: {len(price_path) - 1} 回で、価格 {price_path[-1]:.3f}")

quantity_path = marshall_path(
    lambda q: inverse_demand(q, demand_intercept, demand_slope),
    lambda q: (q - supply_intercept) / supply_slope,
    initial_quantity,
    marshall_step,
)
print(f"マーシャル調整: {len(quantity_path) - 1} 回で、数量 {quantity_path[-1]:.3f}")

## 図: 価格の調整経路

In [ ]:
from lib.chapter05.plot import plot_price_paths

plot_price_paths({"Walras": price_path}, eq_price)

## 試してみよう

- `walras_step` を 0.05 にすると、収束までの回数が 27 回から 65 回に増えます
  （ゆっくり動くので、回数が増えます）
- `walras_step` を 0.5 にすると、価格が収束せず、`RuntimeError` で止まります
  （速すぎると、均衡を行きすぎて、振動が大きくなります）
- `marshall_step` を 0.5 にすると、収束までの回数が 204 回から 33 回に減ります
  （数量は、60 に収束します）